In [ ]:
pip install keras-tuner

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 KB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Trans-Ubiquitination-Colab/BraveHeart

/content/drive/MyDrive/Trans-Ubiquitination-Colab/BraveHeart


In [ ]:
import numpy as np
x = np.load("homo_ubi_embeddings_X.npy")

In [ ]:
x.shape

(4000, 17664)

In [ ]:
y = np.load("homo_ubi_embeddings_Y.npy")

In [ ]:
y.shape

(4000,)

In [ ]:
import pandas as pd
x_dataframe = pd.DataFrame(x)

In [ ]:
x_dataframe

,0,1,2,3,4,5,6,7,8,9,...,17654,17655,17656,17657,17658,17659,17660,17661,17662,17663
0,-0.782214,-0.124671,-0.141697,0.176970,-0.442487,-0.064884,0.649165,0.266145,-0.155975,-0.370200,...,0.031379,0.001916,-0.161234,-0.262236,0.025050,-0.081999,-0.028757,0.067166,-0.265682,-0.092070
1,0.319376,1.104472,-1.257744,-0.986339,-0.664544,-0.130991,1.246551,-0.464423,-0.302342,-0.867527,...,0.094962,0.145508,0.078585,-0.130398,0.396239,-0.295938,-0.349839,0.105124,-0.236944,0.571456
2,-0.904048,-0.229957,-0.156956,0.041050,-0.584102,0.052986,0.524052,0.443857,-0.203537,-0.283925,...,0.083663,0.013869,-0.151100,-0.243959,0.012364,-0.060610,-0.048819,0.060473,-0.241524,-0.111876
3,-0.080643,0.867145,-0.856271,-1.009194,-0.758203,0.091213,0.758182,-0.656529,0.412247,0.546839,...,0.011425,0.197644,0.091389,-0.246717,0.365010,-0.236993,-0.225534,0.142884,-0.033701,0.634908
4,-1.061503,-0.368257,-0.190612,0.019377,-0.253407,0.050136,0.479626,0.392251,-0.335405,-0.304329,...,0.041791,0.019136,-0.168744,-0.211274,0.043320,-0.047533,-0.014310,0.072769,-0.225191,-0.106579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,-0.162565,0.876729,-1.058215,-1.321007,-1.241336,0.094999,1.050081,-0.278553,-0.271534,0.042180,...,-0.049624,0.010590,0.085360,0.111223,0.466308,-0.328402,-0.268422,0.154869,-0.412042,0.472307
3996,-0.656452,-0.281502,-0.145550,-0.186631,-0.320786,-0.005594,0.409154,0.315339,-0.052799,-0.346843,...,0.050526,0.013067,-0.172364,-0.195747,0.044481,-0.075037,-0.033817,0.060396,-0.270067,-0.066649
3997,0.046346,0.838802,-1.047061,-1.139279,-0.841509,0.024932,0.925195,-0.593241,0.133541,0.385612,...,-0.207399,0.233713,0.066568,-0.160079,0.733992,-0.204189,-0.318484,0.306680,-0.173608,0.978293
3998,-0.903575,-0.279092,-0.183277,0.085107,-0.344833,0.073812,0.608879,0.217553,-0.058671,-0.469833,...,0.099563,-0.005592,-0.192161,-0.244181,0.009522,-0.088248,-0.023222,0.038603,-0.293412,-0.047889


In [ ]:
y

array([1, 1, 1, ..., 0, 0, 0])

# 1D CNN

In [ ]:
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, Conv1D, ZeroPadding1D, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state = 42)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5)

def CNN_1D():
    model = Sequential()

    # layer 1
    model.add(Conv1D(8, 3, input_shape=(23*768, 1), activation="relu"))
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.1))

    # layer 2
    model.add(Conv1D(16, 3, activation="relu"))
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.2))

    # Flattening Layer:
    model.add(Flatten())
    model.add(Dense(64, activation="relu"))

    # Last Layer:
    model.add(Dense(2, activation="softmax"))
    model.compile(loss="categorical_crossentropy", optimizer=Adam(lr = 0.00005),
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

model1 = CNN_1D()

a = np.asarray(xtrain).reshape(len(np.asarray(xtrain)),23*768,1)

history1_ = model1.fit(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),23*768,1), utils.to_categorical(ytrain,2),
                    validation_data=(np.asarray(xval).reshape(len(np.asarray(xval)),23*768,1), utils.to_categorical(yval,2)),
                    epochs=50, batch_size=20, verbose=1)    

score = model1.evaluate(np.asarray(xtest).reshape(len(np.asarray(xtest)),23*768,1), utils.to_categorical(ytest,2), verbose=1)
print("categorical_crossentropy loss, Accuracy, Precision, Recall scores:", score)


plt.plot(history1_.history["accuracy"])
plt.plot(history1_.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("1D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()
plt.figure(figsize=(10,10), dpi = 600)

plt.plot(history1_.history["loss"])
plt.plot(history1_.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("1D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()

print(model1.summary())

import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions1 = model1.predict(xtest)
bert2_probs_cnn1 = cnn_predictions1[:,1]
cnn_predictions1 = np.argmax(cnn_predictions1, axis = 1)
confusion_matrix = confusion_matrix(ytest, cnn_predictions1)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 1D CNN Confusion Matrix", fontsize = 20)

plt.show()

from sklearn.metrics import roc_curve
fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp, _ = roc_curve(ytest, bert2_probs_cnn1)

from sklearn.metrics import auc
auc_keras_bert_cnn1_hyp = auc(fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp)
print("AUC Score", auc_keras_bert_cnn1_hyp)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn1_hyp, tpr_keras_bert_cnn1_hyp, label="BERT + 1D-CNN: {:.3f}".format(auc_keras_bert_cnn1_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 1D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")


precision_bert2_cnn1, recall_bert2_cnn1, _ = precision_recall_curve(ytest, bert2_probs_cnn1)
auc_precision_recall_bert2_cnn1 = auc(recall_bert2_cnn1, precision_bert2_cnn1)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2_cnn1, precision_bert2_cnn1, label="BERT + 1D-CNN: {:.3f}".format(auc_precision_recall_bert2_cnn1))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 1D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")

# 2D CNN Without Tuner

In [ ]:
################ Without tuner ########################
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve



xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state=None, shuffle=True)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5, random_state=None, shuffle=True)

def CNN_2D():
    model = Sequential()

    # layer 1
    model.add(Conv2D(32, 4, 4, input_shape=(768, 23, 1), activation="relu")) 
    model.add(MaxPooling2D(2))
    model.add(Dropout(0.1))


    # layer 2
    model.add(Conv2D(16, 2, 2, activation="relu"))
    #model.add(MaxPooling2D(2))
    model.add(Dropout(0.2))

    

    # Flattening Layer:
    model.add(Flatten())
    model.add(Dense(96, activation="relu"))
    model.add(Dense(32, activation="relu"))

    # Last Layer:
    model.add(Dense(2, activation="softmax"))
    model.compile(loss="binary_crossentropy", optimizer=Adam(lr = 0.001),
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model

model2 = CNN_2D()

stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=40,  verbose=1, mode='min')

history2_ = model2.fit(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1), utils.to_categorical(ytrain,2),
                    validation_data=(np.asarray(xval).reshape(len(np.asarray(xval)),768,23,1), utils.to_categorical(yval,2)), callbacks = [stop_early],
                    epochs=200, batch_size=150, verbose=1)


Epoch 1/200
22/22 [==============================] - 1s 23ms/step - loss: 0.6958 - accuracy: 0.4978 - precision_8: 0.4978 - recall_8: 0.4978 - val_loss: 0.6939 - val_accuracy: 0.4975 - val_precision_8: 0.4975 - val_recall_8: 0.4975
Epoch 2/200
22/22 [==============================] - 0s 9ms/step - loss: 0.6931 - accuracy: 0.4947 - precision_8: 0.4947 - recall_8: 0.4947 - val_loss: 0.6935 - val_accuracy: 0.4900 - val_precision_8: 0.4900 - val_recall_8: 0.4900
Epoch 3/200
22/22 [==============================] - 0s 8ms/step - loss: 0.6925 - accuracy: 0.5147 - precision_8: 0.5147 - recall_8: 0.5147 - val_loss: 0.6928 - val_accuracy: 0.5225 - val_precision_8: 0.5225 - val_recall_8: 0.5225
Epoch 4/200
22/22 [==============================] - 0s 9ms/step - loss: 0.6894 - accuracy: 0.5325 - precision_8: 0.5325 - recall_8: 0.5325 - val_loss: 0.6928 - val_accuracy: 0.5300 - val_precision_8: 0.5300 - val_recall_8: 0.5300
Epoch 5/200
22/22 [==============================] - 0s 8ms/step - loss: 0.

In [ ]:
model2.evaluate(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2))

13/13 [==============================] - 0s 4ms/step - loss: 0.6689 - accuracy: 0.6350 - precision_8: 0.6350 - recall_8: 0.6350


[0.6689329743385315,
 0.6349999904632568,
 0.6349999904632568,
 0.6349999904632568]

In [ ]:
plt.plot(history2_.history["accuracy"])
plt.plot(history2_.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Accuracy", fontsize = 15)
plt.title("2D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()


plt.plot(history2_.history["loss"])
plt.plot(history2_.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("2D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()


print(model2.summary())

import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions3 = model2.predict(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1)) # flattened x
bert2_probs = cnn_predictions3[:,1]
print(cnn_predictions3[:,1])    # [prob of 0s, prob of 1s]
cnn_predictions3 = np.argmax(cnn_predictions3, axis = 1)
print(cnn_predictions3)
confusion_matrix = confusion_matrix(ytest, cnn_predictions3)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 2D CNN + H. Tuning Confusion Matrix", fontsize = 20)
plt.show()


from sklearn.metrics import roc_curve
fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, thresholds_keras = roc_curve(ytest, bert2_probs)

from sklearn.metrics import auc
auc_keras_bert_cnn2_hyp = auc(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp)
print("AUC Score", auc_keras_bert_cnn2_hyp)

plt.clf()
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, label="BERT + 2D-CNN: {:.3f}".format(auc_keras_bert_cnn2_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

precision_bert2, recall_bert2, _ = precision_recall_curve(ytest, bert2_probs)
auc_precision_recall_bert2 = auc(recall_bert2, precision_bert2)

plt.clf()
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2, precision_bert2, label="BERT + 2D-CNN: {:.3f}".format(auc_precision_recall_bert2))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()


print("----------------------------------------------")

# 2D CNN with Tuner

In [ ]:
######################################## BERT + 2D CNN PART ############################################

def build_model(hp):
    # create model object
    model = keras.Sequential([

        # adding first convolutional layer
        keras.layers.Conv2D(
            # adding filter
            filters=hp.Int('conv_1_filter', min_value=8, max_value=32, step=2),
            # adding filter size or kernel size
            kernel_size=hp.Choice('conv_1_kernel', values=[2, 6]),
            # activation function
            activation='relu',
            input_shape=(768, 23, 1)),
        
        # adding second convolutional layer
        keras.layers.Conv2D(
            # adding filter
            filters=hp.Int('conv_2_filter', min_value=16, max_value=32, step=4),
            # adding filter size or kernel size
            kernel_size=hp.Choice('conv_2_kernel', values=[2, 8]),
            # activation function
            activation='relu'
        
        ),
        
        # adding flatten layer
        keras.layers.Flatten(),
        # adding dense layer
        keras.layers.Dense(
            units=hp.Int('dense_1_units', min_value=16, max_value=96, step=4),
            activation='relu'
        ),

        # output layer
        keras.layers.Dense(2, activation='softmax')
    ])

    # compilation of model
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[0.00006, 0.00008, 0.0001])),  # 1e-2, 1e-3, 1e-4
                  loss='categorical_crossentropy',
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    model.summary()
    return model

#importing random search
from kerastuner import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters

#creating randomsearch object
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials = 20, directory = "output", project_name = "Ubi_AfterHyperParameterTuning_2D_CNN")


xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size = 0.20, random_state=None, shuffle=True)
xval, xtest, yval, ytest = train_test_split(xtest, ytest, test_size = 0.5, random_state=None, shuffle=True)

stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=40,  verbose=1, mode='min')

tuner.search(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1), utils.to_categorical(ytrain,2), 
             validation_data = (np.asarray(xval.reshape(len(np.asarray(xval)),768,23,1)), utils.to_categorical(yval,2)), epochs = 30, callbacks = [stop_early])


model_2D_ht = tuner.get_best_models(num_models=1)[0]

model_2D_ht.summary()

history3 = model_2D_ht.fit(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2),
                           epochs=200, batch_size = 100, validation_split=0.1,
                           initial_epoch=1)     # epochs=15

In [ ]:
plt.plot(history3.history["accuracy"])
plt.plot(history3.history["val_accuracy"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Accuracy", fontsize = 15)
plt.title("2D CNN Model Accuracy", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()
plt.figure(figsize=(10,10), dpi = 600)

plt.plot(history3.history["loss"])
plt.plot(history3.history["val_loss"])
plt.xlabel("Epoch", fontsize = 15)
plt.ylabel("Loss", fontsize = 15)
plt.title("2D CNN Model Loss", fontsize = 20)
plt.legend(["Train", "Test"], loc="upper left",  prop={"size": 15})
plt.show()

print(model_2D_ht.summary())

import seaborn as sns
from sklearn.metrics import confusion_matrix
cnn_predictions3 = model_2D_ht.predict(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1))
bert2_probs = cnn_predictions3[:,1]
print(cnn_predictions3[:,1])    
cnn_predictions3 = np.argmax(cnn_predictions3, axis = 1)
print(cnn_predictions3)
confusion_matrix = confusion_matrix(ytest, cnn_predictions3)
sns.heatmap(confusion_matrix, annot = True, fmt = "d", cbar = False)
plt.title("BERT + 2D CNN + H. Tuning Confusion Matrix", fontsize = 20)
plt.show()


from sklearn.metrics import roc_curve
print(cnn_predictions3)
fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, thresholds_keras = roc_curve(ytest, bert2_probs)

from sklearn.metrics import auc
auc_keras_bert_cnn2_hyp = auc(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp)
print("AUC Score", auc_keras_bert_cnn2_hyp)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr_keras_bert_cnn2_hyp, tpr_keras_bert_cnn2_hyp, label="BERT + 2D-CNN: {:.3f}".format(auc_keras_bert_cnn2_hyp))
plt.xlabel("False positive rate", fontsize = 15)
plt.ylabel("True positive rate", fontsize = 15)
plt.title("ROC curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")


precision_bert2, recall_bert2, _ = precision_recall_curve(ytest, bert2_probs)
auc_precision_recall_bert2 = auc(recall_bert2, precision_bert2)

plt.plot([0, 1], [0, 1], 'k--')
plt.plot(recall_bert2, precision_bert2, label="BERT + 2D-CNN: {:.3f}".format(auc_precision_recall_bert2))
plt.xlabel("Recall", fontsize = 15)
plt.ylabel("Precision", fontsize = 15)
plt.title("PR curve for BERT + 2D CNN", fontsize = 20)
plt.legend(loc="best",  prop={'size': 15})
plt.show()

print("----------------------------------------------")

# ViT

In [ ]:
pip install tensorflow-addons

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Activation, Conv1D, ZeroPadding1D, MaxPooling1D, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
import tensorflow as tf
from tensorflow import keras
from kerastuner import RandomSearch
from tensorflow.keras import utils
from sklearn.metrics import precision_recall_curve
from tensorflow.keras import layers
import tensorflow_addons as tfa
import tensorflow_addons as tfa
import pickle
import time
from tensorflow.keras.callbacks import TensorBoard
import keras_tuner as kt
from keras_tuner import RandomSearch


num_classes = 2
input_shape = (768, 23, 1)

In [ ]:
learning_rate = 0.001        
weight_decay = 0.0001
batch_size = 200   
num_epochs = 50                                                                        
image_size = 16  
patch_size = 6  
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 12    
transformer_units = [projection_dim * 2, projection_dim, ] 
transformer_layers =  1   
mlp_head_units = [2048, 1024]  

In [ ]:
import keras

#####  Use data augmentation

data_augmentation = keras.Sequential(
    [
        layers.Normalization(),
        layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=0.02),
        layers.RandomZoom(height_factor=0.2, width_factor=0.2),
    ],
    name="data_augmentation",
)
# Compute the mean and the variance of the training data for normalization.
data_augmentation.layers[0].adapt(np.asarray(xtrain).reshape(len(np.asarray(xtrain)),768,23,1))

In [ ]:
#######  Implement multilayer perceptron (MLP)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
#######  Implement patch creation as a layer

class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

In [ ]:
######  Implement the patch encoding layer

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

In [ ]:
# ################################ EDITTING VIT CODE #################################

######  Build the ViT model

def optimal_vit_classifier(hp):
    print("OK-1")
    inputs = layers.Input(shape=input_shape)
    # Augment data.
    augmented = data_augmentation(inputs)
    print(augmented.shape)
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
    print("OK-2")

    #num_heads_hp = hp.Int("num_heads", min_value = 20, max_value = 30, step = 2)                     
    #num_heads_hp = hp.Int("num_heads", min_value = 8, max_value = 8, step = 2) 

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):                                                           
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads = num_heads, key_dim=projection_dim, dropout=0.1                            
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])
        print("OK-3")

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = layers.Dense(num_classes, activation ="softmax")(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    print("OK-4")


    #hp_learning_rate = hp.Choice("learning_rate", values = [0.00008, 0.0001, 0.0003])               
    #hp_weight_decay = hp.Choice("weight_decay", values = [0.0001])                             

    optimizer = tfa.optimizers.AdamW(learning_rate = learning_rate, weight_decay = weight_decay)   

    print("OK-5")
    
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[0.0005, 0.0001])),    
                  loss='categorical_crossentropy',
                  metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC(), 
                           tf.keras.metrics.AUC(curve = "ROC"), tf.keras.metrics.AUC(curve = "PR")])
    
    return model

tuner_ch = kt.RandomSearch(optimal_vit_classifier,
                        objective="val_accuracy",                                                                   
                        max_trials = 15, directory = "output", project_name = "Crot_AfterHyperParameterTuning_ViT_homo")

OK-1
(None, 16, 16, 1)
OK-2
OK-3
OK-4
OK-5


In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.0008, patience=100,  verbose=1, mode='min')

tuner_ch.search(np.asarray(xtrain.reshape(len(np.asarray(xtrain)),768,23,1)), utils.to_categorical(ytrain,2),
                validation_data = (np.asarray(xval.reshape(len(np.asarray(xval)),768,23,1)), utils.to_categorical(yval,2)), epochs = 50, callbacks = [stop_early])

In [ ]:
model3 = tuner_ch.get_best_models(num_models=1)[0]
history3_ = model3.fit(np.asarray(xtest).reshape(len(np.asarray(xtest)),768,23,1), utils.to_categorical(ytest,2), epochs = 700)

OK-1
(None, 16, 16, 1)
OK-2
OK-3
OK-4
OK-5
Epoch 1/700
13/13 [==============================] - 4s 23ms/step - loss: 0.6934 - accuracy: 0.5200 - precision: 0.5200 - recall: 0.5200 - auc: 0.5475 - auc_1: 0.5475 - auc_2: 0.5359
Epoch 2/700
13/13 [==============================] - 0s 22ms/step - loss: 0.6909 - accuracy: 0.5250 - precision: 0.5250 - recall: 0.5250 - auc: 0.5267 - auc_1: 0.5267 - auc_2: 0.5335
Epoch 3/700
13/13 [==============================] - 0s 22ms/step - loss: 0.7006 - accuracy: 0.4700 - precision: 0.4700 - recall: 0.4700 - auc: 0.4981 - auc_1: 0.4981 - auc_2: 0.5008
Epoch 4/700
13/13 [==============================] - 0s 22ms/step - loss: 0.6932 - accuracy: 0.5225 - precision: 0.5225 - recall: 0.5225 - auc: 0.5325 - auc_1: 0.5325 - auc_2: 0.5230
Epoch 5/700
13/13 [==============================] - 0s 23ms/step - loss: 0.6908 - accuracy: 0.5400 - precision: 0.5400 - recall: 0.5400 - auc: 0.5328 - auc_1: 0.5328 - auc_2: 0.5334
Epoch 6/700
13/13 [=======================